## Ingress Configurations
A basic single host multi-path routing looks like:
```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: app-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  ingressClassName: nginx
  rules:
    - host: example.com
      http:
        paths:
          - path: /api
            pathType: Prefix
            backend:
              service:
                name: backend-service
                port:
                  number: 8080
          - path: /
            pathType: Prefix
            backend:
              service:
                name: frontend-service
                port:
                  number: 80
```

`metadata.annotations` is a key value store that contains vendor specific features and behaviours to the Ingress Controller. In this particular case it tells Nginx to rewrite `/api/v1/users` to `/v1/users`.

The `backend.service.port.number` value matches the `spec.ports.port` number in the corresponding service definition.

Ingress Controller can also act as TLS termination point:
```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: app-ingress
  annotations:
    cert-manager.io/cluster-issuer: letsencrypt-prod  # A configured certificate manager
    nginx.ingress.kubernetes.io/ssl-redirect: true    # HTTPS only enforcement
spec:
    ingressClassName: nginx
    tls:
      - hosts: 
          - example.com
        secretName: app-tls
      
```

And a configured Secret:
```yaml
apiVersion: v1
kind: Secret
metadata:
  name: app-tls
type: kubernetes.io/tls
data:
  tls.crt: <base64-encoded fullchain certificate>
  tls.key: <base64-encoded private key>
```